# ⚡ 新能源行业智能体
**一键部署到 Colab Free (T4 GPU) - Gradio 公网访问**

---

## ⚠️ 运行前必须配置 Secrets

**在 Colab 左侧边栏 → 🔑 钥匙图标 → Secrets → 添加以下密钥:**

| Secret 名称 | 值 | 说明 |
|------------|-----|------|
| `HF_TOKEN` | `hf_xxx...` | HuggingFace Token（可选，Qwen 公开模型不需要）|
| `NGROK_TOKEN` | `xxx...` | ngrok Token（可选，注册 https://ngrok.com 免费获取）|

> 📌 所有 Token 通过 Secrets 安全注入，代码中不包含任何硬编码凭证。

---

In [ ]:
# ── Cell 1: 读取 Colab Secrets ──
import os

try:
    from google.colab import userdata
    # 读取 Secrets
    for secret_name in ["HF_TOKEN", "NGROK_TOKEN"]:
        try:
            val = userdata.get(secret_name)
            if val:
                os.environ[secret_name] = val
                print(f"✅ {secret_name} 已加载")
            else:
                print(f"⚠️ {secret_name} 未配置（可选）")
        except Exception:
            print(f"⚠️ {secret_name} 未配置（可选）")
except ImportError:
    print("⚠️ 非 Colab 环境，从环境变量读取")

print("✅ Secrets 加载完成")

In [ ]:
# ── Cell 2: 安装依赖 ──
!pip install -q vllm gradio plotly pandas duckduckgo_search pyngrok huggingface_hub openai httpx requests
print("✅ 依赖安装完成")

In [ ]:
# ── Cell 3: 挂载 Google Drive（模型缓存持久化）──
from google.colab import drive
drive.mount('/content/drive')

import os
model_cache = "/content/drive/MyDrive/models"
os.makedirs(model_cache, exist_ok=True)
print(f"✅ 模型缓存目录: {model_cache}")

In [ ]:
# ── Cell 4: Clone 代码仓库 ──
import os

repo_dir = "/content/new-energy-agent"

if os.path.exists(repo_dir):
    print("仓库已存在，拉取最新代码...")
    %cd {repo_dir}
    !git pull
else:
    print("克隆仓库...")
    !git clone https://github.com/pai-pixel/new-energy-agent.git {repo_dir}
    %cd {repo_dir}

print(f"✅ 代码目录: {os.getcwd()}")
!ls -la src/

In [ ]:
# ── Cell 5: 下载/加载模型 (首次 ~5分钟, 后续 ~10秒) ──
import os
from huggingface_hub import snapshot_download

MODEL_ID = "Qwen/Qwen2.5-3B-Instruct-AWQ"
CACHE_DIR = "/content/drive/MyDrive/models"

# 检查是否已缓存
model_path = os.path.join(CACHE_DIR, "models--Qwen--Qwen2.5-3B-Instruct-AWQ")
if os.path.exists(model_path):
    print(f"✅ 模型已缓存: {model_path}")
    # 找到实际的 snapshots 路径
    import glob
    snapshots = os.path.join(model_path, "snapshots", "*")
    snapshot_dirs = glob.glob(snapshots)
    if snapshot_dirs:
        model_path = snapshot_dirs[0]
        print(f"   使用快照: {model_path}")
else:
    print(f"📥 首次下载模型 {MODEL_ID}...")
    print("   这可能需要 2-5 分钟，请耐心等待")
    model_path = snapshot_download(
        MODEL_ID,
        cache_dir=CACHE_DIR,
        token=os.environ.get("HF_TOKEN"),
        resume_download=True,
        max_workers=4,
    )
    print(f"✅ 模型下载完成: {model_path}")

os.environ["MODEL_PATH"] = model_path

In [ ]:
# ── Cell 6: 启动 vLLM 推理服务（后台）──
import subprocess, os, time

model_path = os.environ.get("MODEL_PATH", "")
if not model_path:
    print("❌ 模型路径未设置，请先运行上一个 Cell")
else:
    print(f"🚀 启动 vLLM 推理服务...")
    print(f"   模型: {model_path}")
    print(f"   端口: 8000")
    
    # 终止旧进程
    !pkill -f "vllm.entrypoints" 2>/dev/null || true
    time.sleep(1)
    
    # 后台启动
    log_file = "/content/vllm.log"
    with open(log_file, "w") as f:
        proc = subprocess.Popen([
            "python", "-m", "vllm.entrypoints.openai.api_server",
            "--model", model_path,
            "--quantization", "awq",
            "--max-model-len", "4096",
            "--gpu-memory-utilization", "0.85",
            "--port", "8000",
            "--host", "0.0.0.0",
        ], stdout=f, stderr=f)
    
    print(f"   vLLM PID: {proc.pid}")
    print(f"   日志: {log_file}")
    print("   ⏳ 等待服务就绪...")
    print("   (可查看日志: !tail -f /content/vllm.log)")

In [ ]:
# ── Cell 7: 等待 vLLM 就绪 + 启动智能体 ──
import os, sys, time

%cd /content/new-energy-agent
sys.path.insert(0, "/content/new-energy-agent")

from src.agent import NewEnergyAgent, create_ui, wait_for_vllm, start_ngrok
import logging
logging.basicConfig(level=logging.INFO)

# 等待 vLLM
if wait_for_vllm(max_retries=30, interval=3):
    print("\n✅ vLLM 已就绪!")
else:
    print("\n❌ vLLM 启动超时，请检查 !tail -50 /content/vllm.log")
    
# 启动 ngrok (可选)
ngrok_url = start_ngrok(7860)
if ngrok_url:
    print(f"\n🔗 ngrok 公网地址: {ngrok_url}")

# 创建 Agent
agent = NewEnergyAgent()

# 启动 Gradio
print("\n🚀 启动 Gradio UI...")
print("=" * 60)

demo = create_ui(agent)
demo.queue(max_size=32).launch(
    server_name="0.0.0.0",
    server_port=7860,
    share=True,
    show_error=True,
)

# Gradio share=True 会自动输出 *.gradio.live 公网 URL

In [ ]:
# ── Cell 8 (可选): 保活脚本 ──
# 运行此 Cell 可延长 Colab 会话时间

import time
from IPython.display import display, Javascript

keep_alive_js = """
function ClickConnect() {
    console.log('Colab 保活心跳: ' + new Date());
    document.querySelector('colab-connect-button').click();
}
setInterval(ClickConnect, 60000);  // 每分钟心跳
"""

display(Javascript(keep_alive_js))
print("✅ 保活脚本已启动（每分钟心跳）")
print("⚠️ 免费版 Colab 通常运行 4-12 小时后自动断开")

---
## 📖 使用说明

### 基本对话
- 「你好」→ 闲聊
- 「上海上网电价」→ 查询电价
- 「北京天气怎么样」→ 查询天气
- 「光伏补贴政策」→ 联网搜索

### 多轮上下文
- 「上海上网电价」→ 查询上海上网电价
- 「江苏呢」→ 自动继承「上网电价」，查江苏
- 「工商业电价呢」→ 自动继承「江苏」，切换电价类型
- 「那天气呢」→ 自动继承「江苏」，切换天气查询

### 快捷按钮
点击底部快捷按钮可以快速填充查询前缀

---
> ⚡ 新能源行业智能助手 | [GitHub](https://github.com/pai-pixel/new-energy-agent)